# GPT-Style Text Generation
## AIAT 122 – Deep Learning

## Learning objectives
- Load a pre-trained GPT-style model and generate text from prompts.
- Control generation with temperature and sampling (top-k, top-p).
- Compare different settings and see sample outputs.

**Where is this used in real life?** GPT-style models power chatbots (e.g. ChatGPT), code completion (e.g. GitHub Copilot), and content creation. **We use a decoder-only Transformer (GPT) to generate coherent text from a prompt. We use it instead of RNNs or n-gram models because** it captures long-range context and produces fluent, context-aware text; RNNs are slower and forget long context, and n-grams don’t model meaning.

**Prerequisites:** Python 3.8+, basics of transformers/attention. Before starting: run the imports cell; if `transformers` or `torch` fails, install with `pip install transformers torch` and restart the kernel.


## Short theory
- **GPT** = Generative Pre-trained Transformer: decoder-only Transformer trained to predict the next token.
- **Text generation:** Given a prompt, the model autoregressively samples the next token; we decode to get text.
- **Temperature:** Higher (e.g. 0.9) → more diverse/creative; lower (e.g. 0.5) → more focused/deterministic.
- **top-k / top-p:** Limit sampling to top-k tokens or nucleus (top-p) to reduce nonsense.
- **Data flow:** prompt → tokenize → model forward (causal attention) → sample next token → append → repeat until max_length or EOS.

**Colab:** Run the pip cell first if using Google Colab. دليل إعداد Google Colab (قم بتشغيل خلية pip أولاً إذا كنت تستخدم Colab)



## Inputs & Outputs
**Inputs:** Hugging Face `transformers` and `torch`, pre-trained GPT-2 (`gpt2`), and text prompts.  
**Outputs:** Loaded model info, generated text for sample prompts, and a comparison of different temperature settings (printed). Run time: under ~10 min (model download once; generation is fast).


In [1]:
%pip install transformers torch datasets -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["USE_TF"] = "0"  # PyTorch only; avoid Keras 3 / TF conflict
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import warnings
warnings.filterwarnings("ignore")
print("✅ Setup complete!")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Setup complete!


<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


### Step 1: Load GPT-2 (pre-trained decoder-only Transformer)


In [3]:
# Load pre-trained GPT-2 (small model). If download fails (e.g. no internet), use a tiny local model so the notebook still runs.
from transformers import GPT2Config

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

try:
    # Prefer cache to avoid re-download; then try download
    model = GPT2LMHeadModel.from_pretrained("gpt2", local_files_only=True)
except (OSError, ValueError):
    try:
        model = GPT2LMHeadModel.from_pretrained("gpt2")
    except (OSError, EnvironmentError) as e:
        print("⚠️ Could not download GPT-2 (network error). Using a tiny GPT-2-style model so you can run the rest of the notebook.")
        print("   For real GPT-2, check your internet and run this cell again.\n")
        config = GPT2Config(
            vocab_size=tokenizer.vocab_size,
            n_positions=128,
            n_embd=256,
            n_layer=4,
            n_head=4,
        )
        model = GPT2LMHeadModel(config)

print(f"Model parameters: {model.num_parameters():,}")
print(f"Vocabulary size: {len(tokenizer)} tokens")
print(f"Max context length: {model.config.n_positions} tokens")
print("\n✅ GPT-2 model loaded!")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

OSError: Can't load the model for 'gpt2'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'gpt2' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

### Step 2: Define generation function and run one example


In [ ]:
# We use GPT here (decoder-only Transformer) for autoregressive text generation; order and context matter.
def generate_text(prompt, model, tokenizer, max_length=80, temperature=0.7, top_k=50, top_p=0.9):
    """Generate text from prompt using temperature and top-k/top-p sampling."""
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# One example
prompt = "The future of artificial intelligence"
out = generate_text(prompt, model, tokenizer, max_length=60, temperature=0.7)
print("Prompt:", prompt)
print("Generated:", out[:400] + "..." if len(out) > 400 else out)
print("\n✅ Generation works!")


### Step 3: Generate from several prompts (e.g. content-creation style)


In [ ]:
# Real-world scenario: short content prompts (marketing / blog style)
prompts = [
    "The impact of artificial intelligence on healthcare",
    "How to get started with machine learning",
]
for p in prompts:
    text = generate_text(p, model, tokenizer, max_length=50, temperature=0.7)
    print(f"Prompt: {p}")
    print(f"Generated: {text[:300]}...")
    print("-" * 50)

### Step 4: Compare different temperatures (quality vs diversity)


In [ ]:
# Compare temperature: lower = more focused, higher = more diverse
import matplotlib.pyplot as plt
prompt = "The impact of artificial intelligence on healthcare"
temps = [0.3, 0.7, 1.0]
lengths = []  # optional: compare output length
for t in temps:
    out = generate_text(prompt, model, tokenizer, max_length=40, temperature=t)
    lengths.append(len(out))
    print(f"Temperature {t}: {out[:200]}...")
    print("-" * 40)
# Simple bar: generated text length vs temperature (higher temp often longer/more varied)
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
ax.bar([str(t) for t in temps], lengths, color=["#2ecc71", "#3498db", "#e74c3c"])
ax.set_xlabel("Temperature")
ax.set_ylabel("Generated length (chars)")
ax.set_title("Generation length vs temperature")
plt.tight_layout()
plt.show()
print("After this cell you should see: three printed samples and one bar chart. Lower temp → often shorter, more repetitive; higher → more varied.")



### Step 5: Fine-tuning concept (no training here; run time stays short)
In real life, companies fine-tune GPT for domain-specific content (legal, medical) or brand voice. Full fine-tuning needs more data and GPU time; here we only show the idea.

In [ ]:
# Conceptual example: Fine-tuning setup
# In production, you would:
# 1. Prepare domain-specific dataset
# 2. Configure training arguments
# 3. Fine-tune model
# 4. Evaluate on test set

print("📚 Fine-tuning Concept:")
print("\n1. Prepare Dataset:")
print(" - Collect domain-specific text (e.g., medical articles)")
print(" - Format as text files or use Hugging Face datasets")
print("\n2. Configure Training:")
print(" - Learning rate: 5e-5")
print(" - Batch size: 4-8 (depending on GPU)")
print(" - Epochs: 3-5")
print("\n3. Fine-tune Model:")
print(" - Use Trainer API from transformers")
print(" - Monitor loss and perplexity")
print("\n4. Evaluate:")
print(" - Test on held-out data")
print(" - Compare with base model")
print("\n✅ Fine-tuning process understood!")

## 🧩 Mini-exercise | تمرين مصغر

**Try it:** Change the temperature (e.g. 0.7 → 0.3 or 1.0) and generate again from the same prompt. How does the output change? Or try a different prompt.

---

## Summary
**What you did**
- Loaded pre-trained GPT-2 and generated text from prompts.
- Used a `generate_text` helper with temperature and top-k/top-p.
- Compared different temperatures and saw a simple length vs temperature plot.

**In real life you'd also:** Fine-tune on domain data, use larger models (e.g. via API), add safety filters, and deploy with batching and caching.

**The main idea:** GPT-style models generate text autoregressively; temperature and sampling controls trade off coherence vs diversity.

**Next:** `08_text_generation_rnn_lstm_gru.ipynb` shows RNN/LSTM/GRU-based text generation for comparison with Transformer-based GPT.